<a href="https://colab.research.google.com/github/riteshbchauhan/python-practice/blob/main/ChatGPT_model_for_Language_Translation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import tensorflow as tf

text_file = tf.keras.utils.get_file(
    fname = 'fra-eng.zip',
    origin = 'http://storage.googleapis.com/download.tensorflow.org/data/fra-eng.zip',
    extract = True,
)

3423204/3423204 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [3]:
import pathlib

text_file = pathlib.Path(text_file).parent / 'fra.txt'

In [9]:
print(text_file)

/root/.keras/datasets/fra.txt


In [17]:
text_file = pathlib.Path(dataset_dir)  / 'fra-eng_extracted' / 'fra.txt'

In [20]:
# Open and read the file
with open(text_file) as fp:
    text_pairs = [line for line in fp]

In [21]:
# Preview the first few translated sentence pairs
for pair in text_pairs[:5]:
    print(pair)

Go.	Va !

Hi.	Salut !

Run!	Cours !

Run!	Courez !

Who?	Qui ?



In [22]:
import unicodedata
import re

def normalize(line):
  line = unicodedata.normalize('NFKC', line.strip().lower())
  line = re.sub(r"^([^ \w])(?!\s)", r"\1", line)
  line = re.sub(r"(\s[^ \w])(?!\s)", r"\1", line)
  line = re.sub(r"(?!\s)([^ \w])$", r"\1", line)
  line = re.sub(r"(?!\s)([^ \w])$\s", r"\1", line)

  eng, fre = line.split('\t')
  fre = '[start]' + fre + '[end]'
  return eng, fre

In [23]:
with open(text_file) as fp:
  test_pairs = [normalize(line) for line in fp]

In [26]:
import random

for _ in range(5):
  print(random.choice(test_pairs))

('i suggest that you try to get some sleep.', '[start]je suggère que tu essaies de prendre un peu de sommeil.[end]')
('where is it?', '[start]où est-elle ?[end]')
('you are a workaholic.', '[start]tu es un maniaque du travail.[end]')
('i ran into a tree.', "[start]j'ai embouti un arbre.[end]")
("your socks don't match.", '[start]tes chaussettes sont dépareillées.[end]')


In [29]:
eng_tokens, fre_tokens = set(), set()
eng_maxlen, fre_maxlen = 0, 0

for eng, fre in test_pairs:
  eng_token, fre_token = eng.split(), fre.split()
  eng_maxlen = max(eng_maxlen, len(eng_token))
  fre_maxlen = max(fre_maxlen, len(fre_token))
  eng_tokens.update(eng_token)
  fre_tokens.update(fre_token)

print(f"Number of unique English words: {len(eng_tokens)}")
print(f"Number of unique French words: {len(fre_tokens)}")
print(f"Max English sentence length: {eng_maxlen}")
print(f"Max French sentence length: {fre_maxlen}")

Number of unique English words: 25365
Number of unique French words: 44581
Max English sentence length: 47
Max French sentence length: 54


In [30]:
import pickle

with open('test_pairs.pickle', 'wb') as fp:
    pickle.dump(test_pairs, fp)

In [31]:
from tensorflow.keras.layers import TextVectorization

import pickle

with open("test_pairs.pickle", "rb") as fp:
    test_pairs = pickle.load(fp)

In [32]:
random.shuffle(text_pairs)

In [33]:
n_val = int(0.15 * len(text_pairs))
n_train = len(text_pairs) - 2 * n_val

train_pair = text_pairs[:n_train]
test_pair = text_pairs[n_train : n_train + n_val]

In [34]:
vocab_en = 10000
vocab_fr = 20000

seq_length = 25

In [36]:
eng_vect = TextVectorization(
    max_tokens = vocab_en,
    standardize = None,
    split = 'whitespace',
    output_mode = 'int',
    output_sequence_length = seq_length,
)

fre_vect = TextVectorization(
    max_tokens = vocab_fr,
    standardize = None,
    split = 'whitespace',
    output_mode = 'int',
    output_sequence_length = seq_length,
)

In [37]:
train_eng = [pair[0] for pair in train_pair]
train_fre = [pair[1] for pair in train_pair]

eng_vect.adapt(train_eng)
fre_vect.adapt(train_fre)

In [40]:
with open('vectorize.pickle', 'wb') as fp:
    data = {'train': train_pair,
            'test': test_pair,
            'eng_vect': eng_vect.get_config(),
            'fre_vect': fre_vect.get_config(),
            'eng_weights': eng_vect.get_weights(),
            'fre_weights': fre_vect.get_weights()
            }
    pickle.dump(data, fp)

In [42]:
with open('vectorize.pickle', 'rb') as fp:
    data = pickle.load(fp)

In [65]:
train_pair = data['train']
test_pair = data['test']

eng_vect = TextVectorization.from_config(data['eng_vect'])
eng_vect.adapt(["dummy text"])
eng_vect.set_weights(data['eng_weights'])

fre_vect = TextVectorization.from_config(data['fre_vect'])
fre_vect.adapt(["dummy text"])
fre_vect.set_weights(data['fre_weights'])

In [66]:
def format_dataset(eng, fre):

  eng = eng_vect(eng)
  fre = fre_vect(fre)

  source = {
      'encode_inp': eng,
      'decode_inp': fre[:, :-1]
  }

  target = fre[:, 1:]

  return(source, target)

In [67]:
def make_dataset(pairs, batch_size=64):
    processed_pairs = []
    for pair in pairs:
        if isinstance(pair, str):
            processed_pairs.append(normalize(pair))
        else:
            processed_pairs.append(pair)

    eng_text, fre_text = zip(*processed_pairs)
    dataset = tf.data.Dataset.from_tensor_slices((list(eng_text), list(fre_text)))
    return dataset.shuffle(2048).batch(batch_size).map(format_dataset).prefetch(16).cache()

In [68]:
train_ds = make_dataset(train_pair)
test_ds = make_dataset(test_pair)
print("Datasets successfully created!")

Datasets successfully created!


In [73]:
for inputs, target in train_ds.take(1):
  print(inputs['encode_inp'].shape)
  print(inputs['encode_inp'][0])
  print(inputs['decode_inp'].shape)
  print(inputs['decode_inp'][0])
  print(target.shape)
  print(target[0])

(64, 25)
tf.Tensor([1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0], shape=(25,), dtype=int64)
(64, 24)
tf.Tensor([1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0], shape=(24,), dtype=int64)
(64, 24)
tf.Tensor([1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0], shape=(24,), dtype=int64)


In [90]:
import tensorflow as tf
import numpy as np

def pos_enc_metrix(L, d, n = 10000):
  assert d%2 == 0
  d2 = d//2
  P = np.zeros((L, d))
  k = np.arange(L).reshape(-1, 1)
  i = np.arange(d2).reshape(1, -1)

  denom = np.power(n, -i/d2)
  args = k *denom

  P[:, ::2] = np.sin(args)
  P[:, 1::2] = np.cos(args)
  return P

In [91]:
class PostitionalEmbedding(tf.keras.layers.Layer):

  def __init__(self, seq_length, vocab_size, embed_dim, **kwargs):
    super().__init__(**kwargs)
    self.seq_length = seq_length
    self.vocab_size = vocab_size
    self.embed_dim = embed_dim

    self.token_embeddings = tf.keras.layers.Embedding(
        input_dim=vocab_size,
        output_dim=embed_dim,
        mask_zero=True
    )
    matrix = pos_enc_metrix(seq_length, embed_dim)
    self.positional_embedding = tf.constant(matrix, dtype='float32')

  def call(self, inputs):
    embedded_tokens = self.token_embeddings(inputs)
    return embedded_tokens + self.positional_embedding

  def compute_mask(self, *args, **kwargs):
    return self.token_embeddings.compute_mask(*args, **kwargs)

  def get_config(self):
    config = super().get_config()
    config.update({
        'seq_length': self.seq_length,
        'vocab_size': self.vocab_size,
        'embed_dim': self.embed_dim
    })
    return config



In [92]:
for inputs, targets in train_ds.take(1):
  print("Input Tensor shape:", inputs['encode_inp'].shape)
  embed_en = PostitionalEmbedding(seq_length, vocab_en, embed_dim=512)
  en_emb = embed_en(inputs['encode_inp'])
  print("Embedded Tensor shape:", en_emb.shape)

Input Tensor shape: (64, 25)
Embedded Tensor shape: (64, 25, 512)


In [93]:
def self_attention(input_shape, prefix='att',mask=False, **kwargs):
  tf.keras.layers.Input(shape=input_shape, dtype='float32', name=f"{prefix}_in1")

  attention = tf.keras.layers.MutltiHeadAttention(name=f"{prefix}_att1", **kwargs)

  norm = tf.keras.layers.LayerNormalization(name=f"{prefix}_norm1")

  add = tf.keras.layers.Add(name=f"{prefix}_add1")

  attout = attention(query = inputs, value = inputs, use_causal_mask = mask)

  ouput = norm(add([inputs, attout]))

  model = tf.keras.Model(inputs = inputs, outputs = output, name = f"{prefix}_att")

  return model

In [94]:
def cross_attention(input_shape, context_shape, prefix = "att", **kwargs):
  context = tf.keras.layers.Input(shape= context_shape, dtype='float32', name=f"{prefix}_ctx2")

  inputs = tf.keras.layers.Input(shape=input_shape, dtype='float32', name= f"{prefix}_in2")

  attention = tf.keras.layers.MultiHeadAttention(name=f"{prefix}_att2", **kwargs)

  norm = tf.keras.layers.LayerNormalization(name = f"{prefix}_norm2")

  add = tf.keras.layers.Add(name="{prefix}_add2")

  attout = attention(query = inputs, key = context, value = context)

  output = norm(add[attout, inputs])

  model = tf.keras.Model(inputs = [context, inputs], output = output, name = f"{prefix}_crs_at")

  return model

In [95]:
def feed_forward(input_shape, model_dim, ff_dim, dropout=1, prefix = "ff"):
  inputs = tf.keras.layers.Input(shape=input_shape, dtype='float32', name= f"{prefix}_in3")

  dense1 = tf.keras.layers.Dense(ff_dim, name=f"{prefix}_ff1", activation='relu')

  dense2 = tf.keras.layers.Dense(model_dim, name=f"{prefix}_ff2")

  drop = tf.keras.layers.Dropout(dropout, name=f"{prefix}_drop")

  add = tf.keras.layers.Add(name="{prefix}_add3")

  ffout = drop(dense2(dense1(inputs)))

  norm = tf.keras.layers.LayerNormalization(name = f"{prefix}_norm3")

  attout = attention(query = inputs, key = context, value = context)

  output = norm(add[attout, inputs])

  model = tf.keras.Model(inputs = inputs, output = output, name = f"{prefix}_ff")

  return model